# GeoMM-Bench — Unified Experiment

**The single notebook for this experiment.** It runs both probes against the
Vilkyciai-22 pilot and writes one results file. It supersedes the earlier
`GeoMM_Code.ipynb` and `GeoMM_Bench_Complete_Evaluation.ipynb` (which split the
work across two backbones and disagreed).

**Question:** can off-the-shelf AI read well-log *display images*?

| Probe | Closes the rebuttal | Approaches |
|---|---|---|
| A — model breadth | "use a real grounding model / VLM, not just CLIP" | text, vision-CLIP, Grounding DINO, BLIP-2, fusion |
| B — modality adding | "give it more visual data / fuse modalities" | text, vision(logs), vision(logs+FWS), multimodal, multimodal+FWS |

**Verified pilot answer (n=11):** only text works (0.727 acc / 0.746 F1);
every vision/grounding/VQA/multimodal/+FWS approach fails (≤ 0.34 F1);
adding Full Wave Sonic makes it worse. The bottleneck is the visual
representation, not the amount of data.

> Probes A and B use different CLIP backbones (A: open-clip/LAION; B: OpenAI CLIP).
> Numbers are reported per-probe and never merged into one row. On 11 samples the
> CLIP text score ranges ~0.73–0.89 across backbones — reported honestly as
> pilot-scale fragility. Conclusions are written by hand from the metrics table;
> there is no auto-generated abstract cell in this notebook (by design).

## 1. Setup

In [ ]:
# Dependencies (uncomment on first run)
# !pip install torch open-clip-torch transformers pdf2image pillow scikit-learn matplotlib
import os, json, warnings
warnings.filterwarnings("ignore")

LOGS_PDF = "vilkyciai22_logs500.pdf"   # source operator displays (not redistributed; see DATASHEET.md)
FWS_PDF  = "vilkyciai22_fws_im_dt.pdf" # Full Wave Sonic display (Probe B)
OUT_JSON = "results/geomm_bench_results.json"
os.makedirs("results", exist_ok=True)

In [ ]:
import torch
def get_device():
    if torch.backends.mps.is_available(): return "mps", torch.float32
    if torch.cuda.is_available():         return "cuda", torch.float16
    return "cpu", torch.float32
DEVICE, DTYPE = get_device()
print("device:", DEVICE)

## 2. Ground truth (11 labelled intervals)

In [ ]:
with open("data/ground_truth.json") as f:
    GT = json.load(f)["intervals"]
print(len(GT), "intervals;", sorted({g['lithology'] for g in GT}))

## 3. Render source PDFs to per-page images

In [ ]:
from pdf2image import convert_from_path
def load_pages(pdf, dpi=150):
    if not os.path.exists(pdf): 
        print("missing:", pdf); return None
    return {i+1: im.convert("RGB") for i, im in enumerate(convert_from_path(pdf, dpi=dpi))}
log_pages = load_pages(LOGS_PDF)
fws_pages = load_pages(FWS_PDF)

## 4. Probe A — model breadth (open-clip / LAION)
Text, vision-CLIP, Grounding DINO, BLIP-2, CLIP fusion. Uses the package modules
so the notebook and the `run_geomm_bench.py` runner share identical code.

In [ ]:
from geomm_bench import baselines as B
from geomm_bench.optional_models import classify_grounding_dino, classify_blip2
from geomm_bench import metrics as M

def run_probe_A(gt, pages):
    y_true = [g["lithology"] for g in gt]
    preds = {k: [] for k in ["text_only","vision_clip","grounding_dino","blip2","fusion"]}
    for g in gt:
        crop = B.crop_to_depth_interval(pages[g["page"]], g["page"], g["start_depth"], g["end_depth"]) if pages else None
        preds["text_only"].append(B.classify_text_only(g["description"])["predicted"])
        if crop is not None:
            preds["vision_clip"].append(B.classify_vision_clip(crop)["predicted"])
            preds["grounding_dino"].append(classify_grounding_dino(crop)["predicted"])
            preds["blip2"].append(classify_blip2(crop)["predicted"])
            preds["fusion"].append(B.classify_multimodal_fusion(crop, g["description"])["predicted"])
    return y_true, preds

yA, pA = run_probe_A(GT, log_pages)

## 5. Probe B — modality adding + FWS (OpenAI CLIP)
text, vision(logs), vision(logs+FWS), multimodal(basic), multimodal(+FWS).

In [ ]:
from geomm_bench.fws_probe import classify_vision_multi_image, classify_multimodal_multi_image

def run_probe_B(gt, log_pages, fws_pages):
    y_true = [g["lithology"] for g in gt]
    preds = {k: [] for k in ["text_only","vision_logs","vision_logs_fws","multimodal_basic","multimodal_full_fws"]}
    for g in gt:
        lc = B.crop_to_depth_interval(log_pages[g["page"]], g["page"], g["start_depth"], g["end_depth"])
        fc = None
        if fws_pages:
            fp = fws_pages.get(g["page"], list(fws_pages.values())[0])
            fc = B.crop_to_depth_interval(fp, g["page"], g["start_depth"], g["end_depth"])
        imgs_fws = [lc] + ([fc] if fc else [])
        preds["text_only"].append(B.classify_text_only(g["description"])["predicted"])
        preds["vision_logs"].append(classify_vision_multi_image([lc])["predicted"])
        preds["vision_logs_fws"].append(classify_vision_multi_image(imgs_fws)["predicted"])
        preds["multimodal_basic"].append(classify_multimodal_multi_image([lc], g["description"])["predicted"])
        preds["multimodal_full_fws"].append(classify_multimodal_multi_image(imgs_fws, g["description"])["predicted"])
    return y_true, preds

yB, pB = run_probe_B(GT, log_pages, fws_pages)

## 6. Score both probes and write the single results file

In [ ]:
def score(y_true, preds):
    out = {}
    for k, yp in preds.items():
        if not yp:
            out[k] = {"macro_f1": None, "accuracy": None, "note": "not run (imagery absent)"}; continue
        _,_,pc = M.per_class_prf(y_true, yp)
        out[k] = {"macro_f1": round(M.macro_f1(y_true, yp),3),
                  "accuracy": round(M.accuracy(y_true, yp),3),
                  "per_class_f1": {c: round(v,3) for c,v in pc.items()}}
    return out

doc = {"benchmark":"GeoMM-Bench","well":"Vilkyciai-22","n_intervals":len(GT),
       "probes":{
         "A_model_breadth": {"_backbone":"open-clip ViT-B-32 laion2b", "results": score(yA, pA)},
         "B_modality_fws":  {"_backbone":"OpenAI clip-vit-base-patch32", "results": score(yB, pB)},
       }}
with open(OUT_JSON,"w") as f: json.dump(doc, f, indent=2)
print("wrote", OUT_JSON)
for pn,p in doc["probes"].items():
    print(f"\n[{pn}] {p['_backbone']}")
    for k,v in p["results"].items(): print(f"  {k:22s} F1={v['macro_f1']} acc={v['accuracy']}")

## 7. Regenerate the figure (from the results file, so it cannot drift)

In [ ]:
import subprocess
subprocess.run(["python","scripts/make_results_figure.py",
                "--results", OUT_JSON,
                "--out","images/Figure2_Results_Comparison.png"])

## 8. Conclusion (written by hand from the numbers above — no template)
Across both probes, only text-only classification succeeds; every off-the-shelf
vision, grounding, VQA, multimodal, and added-modality approach fails on well-log
display interpretation. Adding Full Wave Sonic degrades rather than improves
performance. The failure spans model families *and* input configurations, so the
limitation is the visual representation itself — motivating a domain-adapted
encoder rather than more data or bigger off-the-shelf models.